# Data: download, read, audit and cache

Downloads the pinned UD treebanks, picks out the Nom/Acc nouns, and writes
`results/audit.json` with noun counts, label balance, dropped nouns, subword
fertility, and word-form overlap between splits and languages. It then samples
the training sentences for each seed and caches mBERT features for every target
noun in `data/cache/`.

In [2]:
import hashlib
import json
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path
from statistics import mean

import conllu
import yaml
from transformers import AutoTokenizer

ROOT = Path.cwd()
CONFIG = yaml.safe_load((ROOT / "config.yaml").read_text())
SPLITS = ("train", "dev", "test")


def treebank_path(language: str, split: str) -> Path:
    prefix = CONFIG["treebanks"][language]["prefix"]
    return ROOT / "data" / "raw" / language / f"{prefix}-ud-{split}.conllu"


## Download

Fetches each treebank at the release pinned in `config.yaml`, skipping files
already on disk, and prints a SHA-256 digest for each file.

In [3]:
for language, treebank in CONFIG["treebanks"].items():
    for split in SPLITS:
        path = treebank_path(language, split)
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            url = (f"https://raw.githubusercontent.com/UniversalDependencies/"
                   f"{treebank['repo']}/{treebank['release']}/{path.name}")
            urllib.request.urlretrieve(url, path)
        print(path.name, hashlib.sha256(path.read_bytes()).hexdigest())

hi_hdtb-ud-train.conllu 532b682d47cdeb460bb665f1c66b2405d659153ab2f298c429f06ed45c016386
hi_hdtb-ud-dev.conllu 639f0ace31dbbea778e068cb89c0beaa37d1bfc4ae645dd2cb701747253ab951
hi_hdtb-ud-test.conllu ad5c21d8774847a3a0f5f4c1b85313222f8e5dda8874d6fde6880f9f46b02975
mr_cmupan-ud-train.conllu e8bcaebd7a0ba277777dcb30e2e8fc9470a93bcf57ef4b87d0aebe9fda6da6fa
mr_cmupan-ud-dev.conllu fd03a75728467ce2eb4e14e09e2edadd893ebc0b37b02c172cdbcaf5f4c4492e
mr_cmupan-ud-test.conllu 57f1bac6c2b07da3f8b7366dcbd98e66a8935546257444477c9c9f13a82bd010


## Read the target nouns

In [4]:
def read_sentences(language: str, split: str) -> list[dict]:
    """Return each sentence's words, UPOS tags and targets as (position, label).
    A target is a NOUN whose Case is exactly Nom or Acc; other nouns are skipped.
    """
    sentences = []
    with treebank_path(language, split).open(encoding="utf-8") as stream:
        for sentence in conllu.parse_incr(stream):
            words = [token for token in sentence if isinstance(token["id"], int)]
            targets = [
                (i, token["feats"]["Case"]) for i, token in enumerate(words)
                if token["upos"] == "NOUN" and (token["feats"] or {}).get("Case") in CONFIG["labels"]
            ]
            sentences.append({
                "words": [token["form"] for token in words],
                "upos": [token["upos"] for token in words],
                "targets": targets,
            })
    return sentences

## Analysis of data

In [5]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"])


def nouns(sentences: list[dict]) -> list[tuple]:
    """Return (form, label, next word's UPOS, number of mBERT pieces) for each target."""
    rows = []
    for sentence in sentences:
        word_ids = tokenizer(sentence["words"], is_split_into_words=True).word_ids()
        pieces = Counter(word_id for word_id in word_ids if word_id is not None)
        upos = sentence["upos"] + [None]
        for i, label in sentence["targets"]:
            rows.append((sentence["words"][i], label, upos[i + 1], pieces[i]))
    return rows


report, train_forms, test_forms = {}, {}, {}
for language in CONFIG["treebanks"]:
    report[language] = {}
    for split in SPLITS:
        sentences = read_sentences(language, split)
        rows = nouns(sentences)
        labels = Counter(label for _, label, _, _ in rows)
        report[language][split] = {
            "sentences": len(sentences),
            "nouns": len(rows),
            "dropped_nouns": sum(s["upos"].count("NOUN") for s in sentences) - len(rows),
            "labels": dict(labels),
            "majority": max(labels.values()) / len(rows),
            "followed_by_adp": mean(next_upos == "ADP" for _, _, next_upos, _ in rows),
            "pieces_per_noun": mean(count for _, _, _, count in rows),
        }
        if split == "train":
            train_forms[language] = defaultdict(set)
            for form, label, _, _ in rows:
                train_forms[language][form].add(label)
        if split == "test":
            test_forms[language] = [form for form, _, _, _ in rows]

    # Undecided: the form occurs with both labels in training, so only context can decide.
    seen = train_forms[language]
    report[language]["test"]["undecided_in_train"] = mean(
        len(seen.get(form, ())) == 2 for form in test_forms[language])
    report[language]["test"]["unseen_in_train"] = mean(
        form not in seen for form in test_forms[language])

report["mr"]["test"]["seen_in_hindi_train"] = mean(
    form in train_forms["hi"] for form in test_forms["mr"])

(ROOT / "results").mkdir(exist_ok=True)
(ROOT / "results" / "audit.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")
print(json.dumps(report, ensure_ascii=False, indent=2))

{
  "hi": {
    "train": {
      "sentences": 13306,
      "nouns": 62104,
      "dropped_nouns": 87,
      "labels": {
        "Acc": 33445,
        "Nom": 28659
      },
      "majority": 0.5385321396367384,
      "followed_by_adp": 0.48777856498776245,
      "pieces_per_noun": 2.7141891021512303
    },
    "dev": {
      "sentences": 1659,
      "nouns": 7923,
      "dropped_nouns": 5,
      "labels": {
        "Nom": 3645,
        "Acc": 4278
      },
      "majority": 0.5399469897765998,
      "followed_by_adp": 0.488956203458286,
      "pieces_per_noun": 2.6940552820901176
    },
    "test": {
      "sentences": 1684,
      "nouns": 8025,
      "dropped_nouns": 11,
      "labels": {
        "Nom": 3634,
        "Acc": 4391
      },
      "majority": 0.547165109034268,
      "followed_by_adp": 0.48922118380062307,
      "pieces_per_noun": 2.655327102803738,
      "undecided_in_train": 0.7049221183800624,
      "unseen_in_train": 0.07439252336448599
    }
  },
  "mr": {
    "train"

## Training samples

Only sentences with at least one target noun are sampled. For each seed, the
Hindi sample is 800 random training sentences; its two halves give the
Hindi-halves baseline. The Marathi sample is a random order of all training
sentences: the first n sentences form the size-n sample, so smaller samples are
nested in larger ones.

In [6]:
import numpy as np

CACHE = ROOT / "data" / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

labelled = {
    language: [i for i, sentence in enumerate(read_sentences(language, "train")) if sentence["targets"]]
    for language in CONFIG["treebanks"]
}
samples = {"hi": {}, "mr": {}}
for seed in CONFIG["seeds"]:
    rng = np.random.default_rng(seed)
    samples["hi"][seed] = rng.choice(labelled["hi"], CONFIG["hindi_train_sentences"], replace=False).tolist()
    samples["mr"][seed] = rng.permutation(labelled["mr"]).tolist()
(CACHE / "samples.json").write_text(json.dumps(samples) + "\n")
print({language: len(ids) for language, ids in labelled.items()}, "labelled training sentences")

{'hi': 13162, 'mr': 7185} labelled training sentences


## Feature cache

Runs mBERT once over each sentence and keeps the first- and last-piece vectors
of every target noun at all 13 layers. `data/cache/{language}_{split}_{pooling}.npy`
has shape (layers, nouns, 768), so one layer loads as a single block;
`{language}_{split}_nouns.json` lists the nouns in the same order. Hindi train
is encoded only for the sampled sentences; the Marathi splits are encoded in full.

In [7]:
import time

import torch
from transformers import AutoModel

device = "xpu" if torch.xpu.is_available() else "cpu"
model = AutoModel.from_pretrained(CONFIG["model"]).to(device).eval()
PIECE = {"first": 0, "last": -1}


def cache_features(language: str, split: str, ids: list[int] | None = None, batch_size: int = 32) -> None:
    """Encode the given sentences (default: every sentence with a target) and save their nouns."""
    started = time.time()
    sentences = read_sentences(language, split)
    ids = sorted(ids if ids is not None else [i for i, s in enumerate(sentences) if s["targets"]])
    nouns = [
        {"sentence": i, "word": w, "form": sentences[i]["words"][w], "label": label}
        for i in ids for w, label in sentences[i]["targets"]
    ]
    shape = (model.config.num_hidden_layers + 1, len(nouns), model.config.hidden_size)
    out = {
        pooling: np.lib.format.open_memmap(
            CACHE / f"{language}_{split}_{pooling}.npy", mode="w+", dtype=np.float32, shape=shape)
        for pooling in CONFIG["poolings"]
    }
    row = 0
    for start in range(0, len(ids), batch_size):
        batch = [sentences[i] for i in ids[start:start + batch_size]]
        encoded = tokenizer([s["words"] for s in batch], is_split_into_words=True,
                            padding=True, return_tensors="pt")
        rows, positions = [], {pooling: [] for pooling in out}
        for b, sentence in enumerate(batch):
            word_ids = encoded.word_ids(b)
            for w, _ in sentence["targets"]:
                pieces = [k for k, word_id in enumerate(word_ids) if word_id == w]
                rows.append(b)
                for pooling in out:
                    positions[pooling].append(pieces[PIECE[pooling]])
        with torch.inference_mode():
            states = torch.stack(model(**encoded.to(device), output_hidden_states=True).hidden_states)
        for pooling, array in out.items():
            array[:, row:row + len(rows)] = states[:, rows, positions[pooling]].float().cpu().numpy()
        row += len(rows)
    for array in out.values():
        array.flush()
    (CACHE / f"{language}_{split}_nouns.json").write_text(json.dumps(nouns, ensure_ascii=False) + "\n")
    print(f"{language}/{split}: {len(ids)} sentences, {len(nouns)} nouns, "
          f"{time.time() - started:.0f}s on {device}")


cache_features("hi", "train", sorted({i for ids in samples["hi"].values() for i in ids}))
for split in SPLITS:
    cache_features("mr", split)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


hi/train: 3552 sentences, 16737 nouns, 87s on xpu
mr/train: 7185 sentences, 25569 nouns, 107s on xpu
mr/dev: 930 sentences, 3308 nouns, 15s on xpu
mr/test: 917 sentences, 3391 nouns, 14s on xpu
